# A full business solution

## Now we will take our project from Day 1 to the next level

### BUSINESS CHALLENGE:

Create a product that builds a Brochure for a company to be used for prospective clients, investors and potential recruits.

We will be provided a company name and their primary website.

See the end of this notebook for examples of real-world business applications.

And remember: I'm always available if you have problems or ideas! Please do reach out.

In [35]:
# imports
# If these fail, please check you're running from an 'activated' environment with (llms) in the command prompt
import sys
from pathlib import Path

week1_dir = Path("../../").resolve()
if str(week1_dir) not in sys.path:
    sys.path.insert(0, str(week1_dir))
import os
import json
from dotenv import load_dotenv
from IPython.display import Markdown, display, update_display
from scraper import fetch_website_links, fetch_website_contents
from openai import OpenAI

In [38]:
# Initialize and constants

load_dotenv(override=True)
api_key = os.getenv('OPENAI_API_KEY')

if api_key and api_key.startswith('sk-proj-') and len(api_key)>10:
    print("API key looks good so far")
else:
    print("There might be a problem with your API key? Please visit the troubleshooting notebook!")
    
MODEL = 'gpt-5-nano'
openai = OpenAI()

API key looks good so far


In [39]:
links = fetch_website_links("https://edwarddonner.com")
links

['#wp--skip-link--target',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/proficient/',
 'https://edwarddonner.com/connect-four/',
 'https://edwarddonner.com/outsmart/',
 'https://edwarddonner.com/about-me-and-about-nebula/',
 'https://edwarddonner.com/posts/',
 'https://edwarddonner.com/',
 'https://news.ycombinator.com',
 'https://nebula.io/?utm_source=ed&utm_medium=referral',
 'https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html',
 'https://edwarddonner.com/curriculum/',
 'https://edwarddonner.com/avatar/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-engineer/',
 'https://edwarddonner.com/2026/02/17/ai-coder-vibe-coder-to-agentic-enginee

## First step: Have GPT-5-nano figure out which links are relevant

### Use a call to gpt-5-nano to read the links on a webpage, and respond in structured JSON.  
It should decide which links are relevant, and replace relative links such as "/about" with "https://company.com/about".  
We will use "one shot prompting" in which we provide an example of how it should respond in the prompt.

This is an excellent use case for an LLM, because it requires nuanced understanding. Imagine trying to code this without LLMs by parsing and analyzing the webpage - it would be very hard!

Sidenote: there is a more advanced technique called "Structured Outputs" in which we require the model to respond according to a spec. We cover this technique in Week 8 during our autonomous Agentic AI project.

In [40]:
link_system_prompt = """
You are provided with a list of links found on a webpage.
You are able to decide which of the links would be most relevant to include in a brochure about the company,
such as links to an About page, or a Company page, or Careers/Jobs pages.
You should respond in JSON as in this example:

{
    "links": [
        {"type": "about page", "url": "https://full.url/goes/here/about"},
        {"type": "careers page", "url": "https://another.full.url/careers"}
    ]
}
"""

In [41]:
def get_links_user_prompt(url):
    user_prompt = f"""
Here is the list of links on the website {url} -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

"""
    links = fetch_website_links(url)
    user_prompt += "\n".join(links)
    return user_prompt

In [42]:
print(get_links_user_prompt("https://edwarddonner.com"))


Here is the list of links on the website https://edwarddonner.com -
Please decide which of these are relevant web links for a brochure about the company, 
respond with the full https URL in JSON format.
Do not include Terms of Service, Privacy, email links.

Links (some might be relative links):

#wp--skip-link--target
https://edwarddonner.com/avatar/
https://edwarddonner.com/curriculum/
https://edwarddonner.com/proficient/
https://edwarddonner.com/connect-four/
https://edwarddonner.com/outsmart/
https://edwarddonner.com/about-me-and-about-nebula/
https://edwarddonner.com/posts/
https://edwarddonner.com/
https://news.ycombinator.com
https://nebula.io/?utm_source=ed&utm_medium=referral
https://www.prnewswire.com/news-releases/wynden-stark-group-acquires-nyc-venture-backed-tech-startup-untapt-301269512.html
https://edwarddonner.com/curriculum/
https://edwarddonner.com/avatar/
https://edwarddonner.com/2026/10/14/ai-fde-the-forward-deployed-engineer-agentic-ai-architect/
https://edwarddon

In [44]:
def select_relevant_links(url):
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    return links
    

In [21]:
select_relevant_links("https://edwarddonner.com")

{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'company page', 'url': 'https://edwarddonner.com/'}]}

In [22]:
def select_relevant_links(url):
    print(f"Selecting relevant links for {url} by calling {MODEL}")
    response = openai.chat.completions.create(
        model=MODEL,
        messages=[
            {"role": "system", "content": link_system_prompt},
            {"role": "user", "content": get_links_user_prompt(url)}
        ],
        response_format={"type": "json_object"}
    )
    result = response.choices[0].message.content
    links = json.loads(result)
    print(f"Found {len(links['links'])} relevant links")
    return links

In [23]:
select_relevant_links("https://edwarddonner.com")

Selecting relevant links for https://edwarddonner.com by calling gpt-5-nano
Found 4 relevant links


{'links': [{'type': 'about page',
   'url': 'https://edwarddonner.com/about-me-and-about-nebula/'},
  {'type': 'LinkedIn profile', 'url': 'https://www.linkedin.com/in/eddonner/'},
  {'type': 'Twitter profile', 'url': 'https://twitter.com/edwarddonner'},
  {'type': 'Facebook profile',
   'url': 'https://www.facebook.com/edward.donner.52'}]}

In [25]:
select_relevant_links("https://huggingface.co")

Selecting relevant links for https://huggingface.co by calling gpt-5-nano
Found 15 relevant links


{'links': [{'type': 'brand page', 'url': 'https://huggingface.co/brand'},
  {'type': 'enterprise page', 'url': 'https://huggingface.co/enterprise'},
  {'type': 'pricing page', 'url': 'https://huggingface.co/pricing'},
  {'type': 'blog', 'url': 'https://huggingface.co/blog'},
  {'type': 'careers page', 'url': 'https://apply.workable.com/huggingface/'},
  {'type': 'GitHub', 'url': 'https://github.com/huggingface'},
  {'type': 'Twitter', 'url': 'https://twitter.com/huggingface'},
  {'type': 'LinkedIn', 'url': 'https://www.linkedin.com/company/huggingface/'},
  {'type': 'Discourse / community', 'url': 'https://discuss.huggingface.co/'},
  {'type': 'Endpoints product', 'url': 'https://endpoints.huggingface.co'},
  {'type': 'Docs', 'url': 'https://huggingface.co/docs'},
  {'type': 'Models', 'url': 'https://huggingface.co/models'},
  {'type': 'Datasets', 'url': 'https://huggingface.co/datasets'},
  {'type': 'Spaces', 'url': 'https://huggingface.co/spaces'},
  {'type': 'Learn', 'url': 'https:/

## Second step: make the brochure!

Assemble all the details into another prompt to GPT-5-nano

In [45]:
def fetch_page_and_all_relevant_links(url):
    contents = fetch_website_contents(url)
    relevant_links = select_relevant_links(url)
    result = f"## Landing Page:\n\n{contents}\n## Relevant Links:\n"
    for link in relevant_links['links']:
        result += f"\n\n### Link: {link['type']}\n"
        result += fetch_website_contents(link["url"])
    return result

In [46]:
print(fetch_page_and_all_relevant_links("https://huggingface.co"))

## Landing Page:

Hugging Face – The AI community building the future.

Hugging Face
Models
Datasets
Spaces
Buckets
new
Docs
Enterprise
Pricing
Website
Tasks
HuggingChat
Collections
Languages
Organizations
Community
Blog
Posts
Daily Papers
Hardware
Learn
Discord
Forum
GitHub
Solutions
Team & Enterprise
Hugging Face PRO
Enterprise Support
Inference Providers
Inference Endpoints
Storage Buckets
Log In
Sign Up
The AI community building the future.
The platform where the machine learning community collaborates on models, datasets, and applications.
Explore AI Apps
or
Browse 2M+ models
Trending on
this week
Models
google/embeddinggemma-2
Updated
10 days ago
•
144k
•
2.43k
jialinyyzz/humanizer
Updated
1 day ago
•
75.3k
•
1.77k
Qwen/Qwen-Image-2.1-Turbo
Updated
4 days ago
•
15.3k
•
898
openbmb/MiniCPM-V-4.7-35B-A3B
Updated
about 15 hours ago
•
9.28k
•
804
abenzerps/Qwen-Image-2.1-Uncensored-Quantized
Updated
4 days ago
•
2.42M
•
4.4k
Browse 2M+ models
Spaces
Running
on
Zero
Agents
1.62k
MiniM

In [47]:
brochure_system_prompt = """
You are an assistant that analyzes the contents of several relevant pages from a company website
and creates a short brochure about the company for prospective customers, investors and recruits.
Respond in markdown without code blocks.
Include details of company culture, customers and careers/jobs if you have the information.
"""

# Or uncomment the lines below for a more humorous brochure - this demonstrates how easy it is to incorporate 'tone':

# brochure_system_prompt = """
# You are an assistant that analyzes the contents of several relevant pages from a company website
# and creates a short, humorous, entertaining, witty brochure about the company for prospective customers, investors and recruits.
# Respond in markdown without code blocks.
# Include details of company culture, customers and careers/jobs if you have the information.
# """


In [ ]:
def get_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:5_000] # Truncate if more than 5,000 characters
    return user_prompt

In [49]:
get_brochure_user_prompt("HuggingFace", "https://huggingface.co")

"\nYou are looking at a company called: HuggingFace\nHere are the contents of its landing page and other relevant pages;\nuse this information to build a short brochure of the company in markdown without code blocks.\n\n\n## Landing Page:\n\nHugging Face – The AI community building the future.\n\nHugging Face\nModels\nDatasets\nSpaces\nBuckets\nnew\nDocs\nEnterprise\nPricing\nWebsite\nTasks\nHuggingChat\nCollections\nLanguages\nOrganizations\nCommunity\nBlog\nPosts\nDaily Papers\nHardware\nLearn\nDiscord\nForum\nGitHub\nSolutions\nTeam & Enterprise\nHugging Face PRO\nEnterprise Support\nInference Providers\nInference Endpoints\nStorage Buckets\nLog In\nSign Up\nThe AI community building the future.\nThe platform where the machine learning community collaborates on models, datasets, and applications.\nExplore AI Apps\nor\nBrowse 2M+ models\nTrending on\nthis week\nModels\ngoogle/embeddinggemma-2\nUpdated\n10 days ago\n•\n144k\n•\n2.43k\njialinyyzz/humanizer\nUpdated\n1 day ago\n•\n75.3k

In [ ]:
def create_brochure(company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [51]:
create_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

Hugging Face is the vibrant, global AI community and collaboration platform at the heart of the machine learning revolution. The company empowers machine learning engineers, scientists, and end users by providing a central hub where they can share, explore, and experiment with open-source models, datasets, and applications. Hugging Face fosters an open and ethical AI future by enabling fast-paced innovation through collaboration.

---

## Our Platform & Solutions

- **Models:** Explore over 2 million machine learning models, including state-of-the-art language, vision, and multimodal models.
- **Datasets:** Access and share over 500,000 curated datasets to fuel your AI projects.
- **Spaces:** Deploy and discover ML applications and demos in a user-friendly environment.
- **Buckets:** Secure and scalable storage solutions to support machine learning workflows.
- **Enterprise Support:** Dedicated plans and tools for businesses, including inference endpoints, team collaboration features, and professional support.

The Hugging Face Hub is designed as a place to accelerate your AI projects, from research to production, supporting both individuals and enterprises with a seamless experience.

---

## Company Culture

Hugging Face is deeply community-driven, embracing openness, collaboration, and ethical AI development. The culture encourages:

- **Collaboration:** Building tools and platforms that enable the AI community worldwide to co-create.
- **Innovation:** Constant exploration at the cutting edge of technology by a talented in-house science team.
- **Openness:** Promoting transparent sharing through open-source libraries and datasets.
- **Ethics:** Commitment to building an AI future that respects fairness, safety, and societal benefit.

The company thrives on diversity, curiosity, and a passion for advancing AI to unlock new possibilities responsibly.

---

## Customers & Community

Hugging Face serves a diverse audience:
- Independent machine learning researchers and practitioners.
- Industrial enterprises seeking scalable AI solutions and professional support.
- Educational institutions powering AI learning and experimentation.
- Developers building applications powered by state-of-the-art AI models.
- Enthusiasts contributing to and benefiting from a rich ecosystem of open-source AI resources.

The platform hosts a thriving daily community, including thousands of models, applications, and datasets trending each week, backed by active forums, Discord chats, and ongoing collaborative projects.

---

## Careers at Hugging Face

Hugging Face is always looking for passionate individuals eager to shape the AI future. Career opportunities include roles in:

- Machine Learning Research & Engineering
- Software Engineering and Infrastructure
- Data Science and Data Engineering
- Product Management and Design
- Community Management and Developer Relations
- Enterprise Customer Success and Support

Working at Hugging Face means joining a mission-driven team committed to building the future of open, collaborative, and ethical AI, where your work impacts millions globally.

---

## Connect with Us

- **Website:** https://huggingface.co
- **Community:** Discord, GitHub, Forums
- **Social media:** Twitter, LinkedIn

Join the movement to build the future of AI together with Hugging Face — the home of machine learning collaboration.

---

*Hugging Face — The AI community building the future.*

In [54]:
marathi_brochure_system_prompt= """whatever brouser you will create translate it into Marathi language with Puneri Patya and show it in same way in output"""

In [55]:
def get_marathi_brochure_user_prompt(company_name, url):
    user_prompt = f"""
You are looking at a company called: {company_name}
Here are the contents of its landing page and other relevant pages;
use this information to build a short brochure of the company in markdown without code blocks.\n\n
"""
    user_prompt += fetch_page_and_all_relevant_links(url)
    user_prompt = user_prompt[:2_000] # Truncate if more than 5,000 characters
    return user_prompt

In [56]:
def create_brochure_Marathi (company_name, url):
    response = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": marathi_brochure_system_prompt},
            {"role": "user", "content": get_marathi_brochure_user_prompt(company_name, url)}
        ],
    )
    result = response.choices[0].message.content
    display(Markdown(result))

In [60]:
create_brochure_Marathi('Haldiram','https://www.haldirams.com/')

# हल्दीराम - तुमचा विश्वासू स्वाद साथी

**हल्दीराम्स ऑनलाइन स्टोअर | गोडाई, नावणूक आणि स्नॅक्स खरेदी करा**

---

## आमची खासियत:

- संपूर्ण भारतात होम डिलिव्हरी आणि ₹७९९ वर मोफत शिपिंग
- साधारण वितरण कालावधी: ३-५ कामकाजाचे दिवस (पिन कोडनुसार वेगवेगळा वेळ लागू शकतो)
- सतर्कतेची सूचना: ऑनलाइन पेमेंट संदर्भात फसवणुकीपासून सावध रहा; सुपर स्टॉकिस्ट, डिस्ट्रीब्यूटर, फ्रँचायजी किंवा नोकरीसाठी आम्ही ऑनलाइन पेमेंट स्वीकारत नाही.

---

## आमची उत्पादने:

### गोडाई आणि स्नॅक्स
- गोड पदार्थ
- नावणूक आणि टेस्टी स्नॅक्स
- बेकरी आयटम्स
- पेये
- रेडी टू ईट पदार्थ
- तूप आणि ड्राय फ्रूट्स
- रेडी टू कुक पदार्थ

---

## भेटवस्तू आणि उपहार:

- लग्नाच्या आणि कॉर्पोरेट भेटवस्तू
- तयार उपहार संच (हार्पर्स)
- कुरेटेड आणि खास भेटवस्तू

---

## खास उपहार संच उदाहरण:

### ८ पदार्थांचा खास ड्राय फ्रूट बॉक्स
- ड्राय ब्लूबेरी, मनुका, पाइनॅपल डिलाइट, काजू, बदाम
- वजन: १.०६० kg
- किंमत: ₹२,९०५/-

### ४ पदार्थांचा ड्राय फ्रूट टोकणी
- बदाम, काजू, पिस्ता, मनुका
- वजन: ९४० gm
- किंमत: ₹१,७२५/-

### ८ पदार्थांची आनंदी भेट
- लिंबू भेल, काजू चटपट मसाला, गोल्डन मिश्रण, रोस्टेड चणा लाडू आणि अधिक
- वजन: १०३९ gm
- किंमत: ₹९९०/-

---

## संपर्क आणि मदत:

- ईमेल: [email protected]
- वेबसाईटवरून ऑर्डर ट्रॅक करा आणि नजीकची स्टोर शोधा

---

### हल्दीरामसह प्रेमाने तयार केलेले खाद्यपदार्थ, तुमच्या रूचीनुसार, तुमच्यासाठी!

---

**हल्दीराम्ससोबत स्वाद आणि आनंदाचे अनुभव घरपोच.**

---

**फॉलो करा:**  
[फेसबुक](https://www.facebook.com/haldirams) | [इंस्टाग्राम](https://instagram.com/haldirams) | [ट्विटर](https://twitter.com/haldirams) | [यूट्यूब](https://www.youtube.com/haldirams)

---

**सावधान! फसवणूक टाळा, सुरक्षित ऑनलाइन खरेदी करा.**

**चल, मग काय वाट पाहताय? हल्दीरामकडे भेट द्या आणि स्वादिष्टतेचा आनंद घ्या!**

--- 

*आपल्या महाराष्ट्राला हल्दीरामचा खास 'पुणेरी पट्या' शैलीत हार्दिक अभिवादन!*

## Finally - a minor improvement

With a small adjustment, we can change this so that the results stream back from OpenAI,
with the familiar typewriter animation

In [52]:
def stream_brochure(company_name, url):
    stream = openai.chat.completions.create(
        model="gpt-4.1-mini",
        messages=[
            {"role": "system", "content": brochure_system_prompt},
            {"role": "user", "content": get_brochure_user_prompt(company_name, url)}
          ],
        stream=True
    )    
    response = ""
    display_handle = display(Markdown(""), display_id=True)
    for chunk in stream:
        response += chunk.choices[0].delta.content or ''
        update_display(Markdown(response), display_id=display_handle.display_id)

In [53]:
stream_brochure("HuggingFace", "https://huggingface.co")

# Hugging Face Brochure

---

## About Hugging Face

**Hugging Face** is the vibrant AI community and collaboration platform that is shaping the future of machine learning. It serves as a central hub where machine learning enthusiasts, engineers, scientists, and organizations come together to create, share, explore, and experiment with open-source AI models, datasets, and applications.

At its core, Hugging Face empowers the next generation of machine learning professionals and end users to build an open and ethical AI future collaboratively.

---

## What We Offer

- **2M+ Open-Source Models:** A vast and rapidly growing library of AI models that cater to a variety of machine learning and natural language processing tasks.
  
- **500k+ Datasets:** Easy access to a broad collection of datasets for training, validating, and benchmarking models.
  
- **Spaces:** Host and deploy interactive AI applications in a collaborative environment.
  
- **Buckets:** Scalable and secure storage solutions tailored for machine learning workflows.
  
- **HuggingChat:** Cutting-edge conversational AI tools.
  
- **Hugging Face PRO & Enterprise Plans:** Premium support, inference endpoints, and dedicated resources for organizations needing enterprise-grade AI infrastructure and assistance.

---

## Company Culture & Community

- **Collaborative Spirit:** Hugging Face fosters a strong community-driven approach. Users, researchers, and developers collaborate openly to accelerate AI development and adoption.
  
- **Open & Ethical AI:** The company is deeply committed to building an open ecosystem with transparency and ethical considerations at the forefront.
  
- **Cutting-Edge Innovation:** The science team at Hugging Face explores the edge of AI and machine learning technology and contributes to some of the most popular open-source ML libraries.
  
- **Inclusive & Supportive:** The platform provides resources such as a documentation hub, a Discord server, forums, and a blog to support learners and experts alike.

---

## Customers & Use Cases

Hugging Face serves a diverse audience including:

- **Individual ML Researchers & Practitioners:** Access to powerful models and datasets for experimentation and learning.
  
- **Businesses & Enterprises:** Enterprise-grade AI solutions to deploy scalable models and AI applications with professional support.
  
- **Academic Institutions:** Tools for research and teaching in AI and machine learning fields.
  
- **AI Startups:** Rapid prototyping and deployment platforms fueling innovation.
  
---

## Careers at Hugging Face

Hugging Face is continuously expanding its talented team. They offer open roles in AI research, software engineering, product development, community management, and enterprise support.

- Join a mission-driven company at the forefront of AI technology.
- Work with a global and diverse team passionate about open-source innovation.
- Contribute to projects that impact millions of AI users worldwide.

Find current openings and apply at their [Careers Page](https://huggingface.co/careers).

---

## Brand Assets & Identity

- Recognized by its friendly **orange and yellow palette** with official colors:  
  - #FFD21E (yellow)  
  - #FF9D00 (orange)  
  - #6B7280 (gray)  

- Downloadable logos and brand materials facilitate seamless partnership and presence across projects.

Explore brand resources at their [Brand Assets Page](https://huggingface.co/brand).

---

## Join the AI Revolution with Hugging Face

Discover, create, and collaborate on cutting-edge AI models and datasets at [huggingface.co](https://huggingface.co).

**Connect with the community:**  
- Discord & Forum for discussions  
- GitHub for source code & contributions  
- Blog & Daily Papers for latest AI research updates  

---

*Hugging Face — Building the AI community of the future together.*

In [ ]:
# Try changing the system prompt to the humorous version when you make the Brochure for Hugging Face:

stream_brochure("HuggingFace", "https://huggingface.co")

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/business.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#181;">Business applications</h2>
            <span style="color:#181;">In this exercise we extended the Day 1 code to make multiple LLM calls, and generate a document.

This is perhaps the first example of Agentic AI design patterns, as we combined multiple calls to LLMs. This will feature more in Week 2, and then we will return to Agentic AI in a big way in Week 8 when we build a fully autonomous Agent solution.

Generating content in this way is one of the very most common Use Cases. As with summarization, this can be applied to any business vertical. Write marketing content, generate a product tutorial from a spec, create personalized email content, and so much more. Explore how you can apply content generation to your business, and try making yourself a proof-of-concept prototype. See what other students have done in the community-contributions folder -- so many valuable projects -- it's wild!</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/important.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#900;">Before you move to Week 2 (which is tons of fun)</h2>
            <span style="color:#900;">Please see the week1 EXERCISE notebook for your challenge for the end of week 1. This will give you some essential practice working with Frontier APIs, and prepare you well for Week 2.</span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/resources.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#f71;">A reminder on 3 useful resources</h2>
            <span style="color:#f71;">1. The resources for the course are available <a href="https://edwarddonner.com/2024/11/13/llm-engineering-resources/">here.</a><br/>
            2. I'm on LinkedIn <a href="https://www.linkedin.com/in/eddonner/">here</a> and I love connecting with people taking the course!<br/>
            3. I'm trying out X/Twitter and I'm at <a href="https://x.com/edwarddonner">@edwarddonner<a> and hoping people will teach me how it's done..  
            </span>
        </td>
    </tr>
</table>

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">Finally! I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a MASSIVE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>